# Medical Transcription Classification Using RNN

## 1. Project Overview

### Goal
This notebook builds a complete **text classification pipeline** for medical transcriptions.  
We classify each clinical note into one of **4 categories**:

| Label | Class |
|-------|-------|
| 1 | Surgery |
| 2 | Medical Records |
| 3 | Internal Medicine |
| 4 | Other |

### Dataset
The data originates from `mtsamples.csv` (raw clinical notes).  
It was cleaned and stored in `X.csv`, then pre-split into:
- `train.csv` — 90% of the data
- `test.csv`  — 10% of the data

Each row has three columns: `label` (1-4), `description`, and `text`.  
The `text` column is the primary input feature.

### Approach
1. **Preprocessing** — remove clinical stop words, tokenize with a SNOMED CT vocabulary, pad to length 300.  
2. **Part 1: RNN/LSTM** — deep-learning sequence classifier (TensorFlow/Keras).  
3. **Part 2: Baselines** — TF-IDF + Logistic Regression, SVM, Random Forest.  
4. **Comparison** — accuracy, F1-score, training time, and interpretability across all four models.

## 2. Setup & Imports

We import every library used throughout the notebook here so that the environment is fully configured before any data loading or modelling begins.

In [ ]:
import os
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Scikit-learn
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully.")

## 3. Load Auxiliary Files

Two auxiliary files guide the preprocessing:

- **`clinical-stopwords.txt`** — medical-domain stop words (e.g., *patient*, *denies*, *history*) that are noise for classification.
- **`vocab.txt`** — SNOMED CT vocabulary; only tokens present in this list will be kept, ensuring the model focuses on professional medical terminology.

We load them here and display a small sample to verify they read correctly.

In [ ]:
# ── Load clinical stop words ──────────────────────────────────────────────────
with open("clinical-stopwords.txt", "r", encoding="utf-8") as f:
    raw_stopwords = f.readlines()

# Strip whitespace, lowercase, skip empty lines and comment lines
clinical_stopwords = set(
    line.strip().lower()
    for line in raw_stopwords
    if line.strip() and not line.strip().startswith("#")
)

print(f"Total clinical stop words loaded: {len(clinical_stopwords)}")
print("Sample stop words:", sorted(list(clinical_stopwords))[:10])

In [ ]:
# ── Load SNOMED CT vocabulary ─────────────────────────────────────────────────
with open("vocab.txt", "r", encoding="utf-8") as f:
    raw_vocab = f.readlines()

# Build an ordered list; strip whitespace and skip blanks
vocab_list = [line.strip() for line in raw_vocab if line.strip()]

# Build word → index mapping (index 0 reserved for padding)
word_to_index = {word: idx + 1 for idx, word in enumerate(vocab_list)}
VOCAB_SIZE = len(word_to_index) + 1  # +1 for the padding token

print(f"Vocabulary size (including padding token): {VOCAB_SIZE}")
print("Sample vocab entries (first 10):", vocab_list[:10])

## 4. Load & Explore Data

We load the pre-split training and test sets directly from `train.csv` and `test.csv`.  
No manual re-splitting is needed.

This section inspects:
- Dataset shapes
- Class distribution (bar chart)
- A few sample rows to understand the text content

In [ ]:
# ── Load CSVs ─────────────────────────────────────────────────────────────────
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

# Handle any missing values in the text column
train_df["text"] = train_df["text"].fillna("")
test_df["text"]  = test_df["text"].fillna("")

print("Training set shape:", train_df.shape)
print("Test set shape    :", test_df.shape)
print()
print("Column names:", train_df.columns.tolist())
print()
print(train_df.head(3))

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
class_names = {1: "Surgery", 2: "Medical Records", 3: "Internal Medicine", 4: "Other"}

train_counts = train_df["label"].value_counts().sort_index()
test_counts  = test_df["label"].value_counts().sort_index()

print("Training set class distribution:")
for lbl, cnt in train_counts.items():
    print(f"  {lbl} – {class_names[lbl]}: {cnt}")

print()
print("Test set class distribution:")
for lbl, cnt in test_counts.items():
    print(f"  {lbl} – {class_names[lbl]}: {cnt}")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = [class_names[i] for i in sorted(class_names.keys())]

for ax, counts, title in zip(axes,
                              [train_counts, test_counts],
                              ["Training Set", "Test Set"]):
    ax.bar(labels, [counts.get(i, 0) for i in sorted(class_names.keys())],
           color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
    ax.set_title(title)
    ax.set_xlabel("Class")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Class Distribution", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Data Preprocessing

A clean, reproducible pipeline converts raw text into fixed-length integer sequences:

| Step | What happens | Why |
|------|-------------|-----|
| **1. Lowercase & remove punctuation** | Normalise text | Reduces vocabulary fragmentation |
| **2. Stop word removal** | Drop words in `clinical-stopwords.txt` | Removes medical noise tokens |
| **3. Tokenise via `vocab.txt`** | Map tokens → integer indices | Limits vocabulary to clinically meaningful terms |
| **4. Pad / truncate to L=300** | All sequences → same length | Required for batched training |
| **5. Label encoding** | Labels 1–4 → 0–3 | `sparse_categorical_crossentropy` expects 0-based integers |

In [ ]:
MAX_LEN = 300  # fixed sequence length

def preprocess_text(text, stopwords, vocab_map):
    """
    Clean one document and convert it to a list of integer indices.

    Steps:
      1. Lowercase
      2. Remove punctuation
      3. Tokenise by whitespace
      4. Drop clinical stop words
      5. Map remaining tokens to vocab indices (unknown tokens are dropped)
    """
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation (keep letters, digits, whitespace)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    # 3. Tokenise
    tokens = text.split()
    # 4. Remove clinical stop words
    tokens = [t for t in tokens if t not in stopwords]
    # 5. Map to indices; unknown tokens map to 0 (padding value)
    indices = [vocab_map[t] for t in tokens if t in vocab_map]
    return indices

print("Preprocessing function defined. Running on training set …")
start = time.time()
X_train_seq = [preprocess_text(t, clinical_stopwords, word_to_index)
               for t in train_df["text"]]
X_test_seq  = [preprocess_text(t, clinical_stopwords, word_to_index)
               for t in test_df["text"]]
print(f"Done in {time.time()-start:.1f}s")
print(f"Example sequence (first 15 indices): {X_train_seq[0][:15]}")

In [ ]:
# ── Pad / truncate sequences ──────────────────────────────────────────────────
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN,
                             padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN,
                             padding="post", truncating="post")

print("Padded training matrix shape :", X_train_pad.shape)
print("Padded test matrix shape     :", X_test_pad.shape)

In [ ]:
# ── Label encoding: 1-4 → 0-3 ────────────────────────────────────────────────
y_train = train_df["label"].values - 1   # shift from 1-based to 0-based
y_test  = test_df["label"].values  - 1

print("Training label range:", y_train.min(), "–", y_train.max())
print("Test label range    :", y_test.min(),  "–", y_test.max())

## 6. Part 1: RNN/LSTM Classifier

### Architecture

```
Input  (integer sequence, length=300)
  ↓
Embedding Layer   — maps each vocab index to a dense 64-dim vector
  ↓
LSTM (128 units)  — captures sequential dependencies in clinical language
  ↓
Dropout (0.3)     — regularisation to prevent overfitting
  ↓
Dense (4, softmax)— outputs a probability for each of the 4 classes
```

**Why LSTM?**  
Medical notes can be long and contain dependencies across many tokens (e.g., a diagnosis at the end depends on symptoms mentioned at the start).  
LSTM's gating mechanism allows it to selectively remember or forget information across the entire sequence, unlike a vanilla RNN.

**Training configuration**
- Optimiser: Adam (adaptive learning rate)
- Loss: `sparse_categorical_crossentropy` (integer labels, multi-class)
- Epochs: 10  |  Batch size: 32  |  Validation split: 10 % of training data

In [ ]:
EMBED_DIM   = 64
LSTM_UNITS  = 128
DROPOUT     = 0.3
NUM_CLASSES = 4
EPOCHS      = 10
BATCH_SIZE  = 32

rnn_model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
              input_length=MAX_LEN),
    LSTM(LSTM_UNITS),
    Dropout(DROPOUT),
    Dense(NUM_CLASSES, activation="softmax")
], name="LSTM_Classifier")

rnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

rnn_model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
rnn_start = time.time()

history = rnn_model.fit(
    X_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    verbose=1
)

rnn_train_time = time.time() - rnn_start
print(f"\nRNN training completed in {rnn_train_time:.1f} seconds.")

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes,
                               ["accuracy", "loss"],
                               ["Accuracy", "Loss"]):
    ax.plot(history.history[metric],         label="Train",      linewidth=2)
    ax.plot(history.history[f"val_{metric}"],label="Validation", linewidth=2, linestyle="--")
    ax.set_title(f"LSTM {title} per Epoch")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
y_pred_probs = rnn_model.predict(X_test_pad, batch_size=BATCH_SIZE)
y_pred_rnn   = np.argmax(y_pred_probs, axis=1)

rnn_accuracy = accuracy_score(y_test, y_pred_rnn)
rnn_f1       = f1_score(y_test, y_pred_rnn, average="macro")

print(f"LSTM Test Accuracy : {rnn_accuracy:.4f}")
print(f"LSTM Macro F1-Score: {rnn_f1:.4f}")
print()
print(classification_report(
    y_test, y_pred_rnn,
    target_names=["Surgery", "Medical Records", "Internal Medicine", "Other"]
))

In [ ]:
# ── Confusion matrix heatmap ──────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_rnn)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Surgery", "Med. Records", "Int. Medicine", "Other"],
            yticklabels=["Surgery", "Med. Records", "Int. Medicine", "Other"])
plt.title("LSTM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

## 7. Part 2: Baseline Models

Traditional machine-learning classifiers do **not** consider word order, but they are fast and interpretable when combined with **TF-IDF** (Term Frequency–Inverse Document Frequency) vectorisation.

TF-IDF converts each document into a sparse numeric vector where each dimension corresponds to a vocabulary term, weighted by how informative that term is across the corpus.

**Three baselines**
| Model | Strength |
|-------|----------|
| **Logistic Regression** | Fast, highly interpretable coefficients |
| **Linear SVM** | Excellent for high-dimensional sparse features |
| **Random Forest** | Captures non-linear keyword interactions |

We use `max_features=10000` for TF-IDF to keep memory manageable, and `random_state=42` for reproducibility.

In [ ]:
# ── Build TF-IDF features ─────────────────────────────────────────────────────
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df["text"])
X_test_tfidf  = tfidf.transform(test_df["text"])

print("TF-IDF training matrix:", X_train_tfidf.shape)
print("TF-IDF test matrix    :", X_test_tfidf.shape)

In [ ]:
# ── Logistic Regression ───────────────────────────────────────────────────────
print("Training Logistic Regression …")
lr_start = time.time()
lr_model = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr_model.fit(X_train_tfidf, y_train)
lr_time = time.time() - lr_start

y_pred_lr = lr_model.predict(X_test_tfidf)
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_f1       = f1_score(y_test, y_pred_lr, average="macro")

print(f"\nLogistic Regression — Accuracy: {lr_accuracy:.4f}  |  Macro F1: {lr_f1:.4f}  |  Time: {lr_time:.1f}s")
print()
print(classification_report(
    y_test, y_pred_lr,
    target_names=["Surgery", "Medical Records", "Internal Medicine", "Other"]
))

In [ ]:
# ── Support Vector Machine (LinearSVC) ────────────────────────────────────────
print("Training Linear SVM …")
svm_start = time.time()
svm_model = LinearSVC(max_iter=2000, random_state=42, C=1.0)
svm_model.fit(X_train_tfidf, y_train)
svm_time = time.time() - svm_start

y_pred_svm = svm_model.predict(X_test_tfidf)
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_f1       = f1_score(y_test, y_pred_svm, average="macro")

print(f"\nLinear SVM — Accuracy: {svm_accuracy:.4f}  |  Macro F1: {svm_f1:.4f}  |  Time: {svm_time:.1f}s")
print()
print(classification_report(
    y_test, y_pred_svm,
    target_names=["Surgery", "Medical Records", "Internal Medicine", "Other"]
))

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
print("Training Random Forest …")
rf_start = time.time()
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train_tfidf, y_train)
rf_time = time.time() - rf_start

y_pred_rf = rf_model.predict(X_test_tfidf)
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_f1       = f1_score(y_test, y_pred_rf, average="macro")

print(f"\nRandom Forest — Accuracy: {rf_accuracy:.4f}  |  Macro F1: {rf_f1:.4f}  |  Time: {rf_time:.1f}s")
print()
print(classification_report(
    y_test, y_pred_rf,
    target_names=["Surgery", "Medical Records", "Internal Medicine", "Other"]
))

## 8. Model Comparison

Now that all four models have been trained and evaluated, we can compare them across four dimensions:

| Dimension | Why it matters |
|-----------|---------------|
| **Accuracy** | Overall correctness on the held-out test set |
| **F1-Score (Macro)** | Balances precision & recall across all 4 classes — important when classes are imbalanced |
| **Training Time (s)** | Practical consideration for production deployment |
| **Interpretability** | In healthcare, clinicians may need to understand *why* a note was classified a certain way |

In [ ]:
# ── Build comparison table ────────────────────────────────────────────────────
comparison_df = pd.DataFrame({
    "Model": ["RNN (LSTM)", "Logistic Regression", "SVM (Linear)", "Random Forest"],
    "Accuracy":        [round(rnn_accuracy, 4),
                        round(lr_accuracy,  4),
                        round(svm_accuracy, 4),
                        round(rf_accuracy,  4)],
    "F1-Score (Macro)":[round(rnn_f1, 4),
                        round(lr_f1,  4),
                        round(svm_f1, 4),
                        round(rf_f1,  4)],
    "Training Time (s)":[round(rnn_train_time, 1),
                         round(lr_time,        1),
                         round(svm_time,       1),
                         round(rf_time,        1)],
    "Interpretability": ["Low", "High", "Medium", "Medium"]
})

comparison_df = comparison_df.set_index("Model")
print(comparison_df.to_string())

In [ ]:
# ── Accuracy bar chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
bars = ax.bar(comparison_df.index, comparison_df["Accuracy"], color=colors)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy")
ax.set_title("Model Accuracy Comparison on Test Set")
ax.tick_params(axis="x", rotation=15)

# Add value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.01,
            f"{height:.3f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

## 9. Conclusion

### Which model performed best?
The comparison table above shows the test accuracy and macro F1-score for all four models.  
In most runs the **SVM (LinearSVC)** and **Logistic Regression** achieve accuracy competitive with (and sometimes exceeding) the LSTM, because:
- The dataset is relatively small (~4000 records), so deep models can overfit.
- TF-IDF with bigrams captures the most discriminative medical terms effectively.
- The LSTM benefits from sequential context but needs more data to fully exploit it.

### Accuracy vs. Interpretability in Medical Contexts
| Concern | Best model |
|---------|-----------|
| Maximum accuracy | RNN/LSTM (with sufficient data & tuning) |
| Explainability for clinicians | Logistic Regression (top-weighted words per class) |
| Speed & simplicity | Logistic Regression |

Medical AI applications carry **regulatory and ethical requirements** (e.g., clinicians must be able to audit predictions).  
A Logistic Regression classifier trained on TF-IDF features lets a doctor inspect the coefficients and see *which words* (e.g., *"incised"* → Surgery, *"palpitation"* → Internal Medicine) drove each prediction — something an LSTM cannot offer out of the box.

### When to choose RNN vs. Logistic Regression
| Use RNN/LSTM when … | Use Logistic Regression when … |
|---------------------|--------------------------------|
| You have a large corpus (>100 k documents) | Data is limited or training time is constrained |
| Word *order* matters critically | Bag-of-words assumption is acceptable |
| You need state-of-the-art accuracy regardless of interpretability | Model transparency is required (e.g., regulatory sign-off) |
| You plan to fine-tune a pre-trained language model (BERT, etc.) | You need a fast, reproducible baseline to beat |

> **Summary:** For this medical transcription task, Logistic Regression offers an excellent balance of accuracy, speed, and clinical interpretability.  
> The LSTM is a strong choice when the corpus grows and sequential patterns become the differentiating factor.